### Latin square Simulation

- Latin square is the square with all rows and columns has distinct symbol: https://en.wikipedia.org/wiki/Latin_square
- In this notebook, the author will simulate the latent square for reasonable dimensions <= 11 and provide an est for total number of latent squares using SMC
- Additionally, the same idea will be applied to solve a sudoku puzzle. 

Let's start with some setup! Define:  
- $\pi_0$ is the uniform distribution of all squares n x n with each row is already a permutation from $1 - n$
so that we only need to consider the validity of columns.

- $\pi_1$ is the uniform distribution of all Latin squares n x n.

We will start from $\pi_0$ to generate samples from $\pi_T$

- Let's say if we directly apply the ideas of SMCS by building a list of intermediate distributions $\pi_0^{(1 - \lambda)} * \pi_1^{\lambda}$, what happen is that for all lambdas > 0, all intermediate dists will be collapsed to the target distribution as its pdf is exactly the same aft normalizing (i.e. = 0 if the square is non-Latin and = 1 otherwise)

The idea is interesting as we will try to not directly sample from our target dist but from an "approximate" version of it. 
Let's consider: 

$target = e^{-lambda * score(X)}$ with lambda is big enough value let say 1000. 

- Score(X) here is a score function to measure how "bad" a sample from a Latin properties.
- When X is Latin square, score(X) will be zero and target = 1, otherwise, this value is very small, thanks to the big lambda. By doing SMC on this alternative dist, most of the samples we received will expect to be Latin.

In [1]:
from libs import MCMC, SMC, AcceptanceTracker

First of all, define a scorer class to maintain all scorer methods we will have 

In [2]:
from jax import numpy as jnp

import numpy as np 

class Scorer:
    @staticmethod
    def score_repetitive_columns(flat_board: jnp.array):
        n = np.sqrt(len(flat_board)).astype(int)
        assert len(flat_board) == n * n
        
        score = 0 
        for c in range(n):
            # symbols are 1..n, so the mask needs n + 1 slots
            seen = jnp.zeros(n + 1, dtype=bool)
            for x in flat_board[c::n]:
                score += jnp.where(seen[x], 1, 0)
                seen = seen.at[x].set(True)
        return score 

    @staticmethod
    def score_repetitive_rows(flat_board: jnp.array):
        n = np.sqrt(len(flat_board)).astype(int)
        assert len(flat_board) == n * n
        
        score = 0 
        for r in range(n):
            # symbols are 1..n, so the mask needs n + 1 slots
            seen = jnp.zeros(n + 1, dtype=bool)
            for x in flat_board[r * n : r * n + n : ]:
                score += jnp.where(seen[x], 1, 0)
                seen = seen.at[x].set(True)
        return score 

    @staticmethod
    def score_sudoku(flat_board: jnp.array):
        assert len(flat_board) == 9 * 9

        score_cols = Scorer.score_repetitive_columns(flat_board) 
        score_rows = Scorer.score_repetitive_rows(flat_board) 
        return score_cols + score_rows

assert Scorer.score_repetitive_columns(jnp.array([1,2,1,2])) == 2
assert Scorer.score_repetitive_columns(jnp.array([1,2,2,1])) == 0

Declare an abstract class for any solver

In [3]:
from abc import ABC, abstractmethod
import jax

class ISolver(ABC):
    def __init__(self, n: int, target_dist_logpdf, prior_dist_logpdf, proposed_fn):
        self.__n = n
        self.__key = random.key(100)
        self.smc = SMC(
            dims = n * n, 
            target_dist_logpdf = target_dist_logpdf, 
            prior_dist_logpdf = prior_dist_logpdf, 
            proposed_fn = proposed_fn, 
            key = self._split_key()[0]
        )
        self.log_z = 0.0

    def _split_key(self, n = 2):
        split_keys = random.split(self.__key, n)
        self.__key = split_keys[0]
        return split_keys[1:]

    @abstractmethod
    def _generate_initial_state(self, key):
        pass

    def _generate_initial_states(self, no_samples: int):
        sub_keys = self._split_key(no_samples + 1)
        states = jax.vmap(
            lambda key: self._generate_initial_state(key)
        )(sub_keys)
        return states

    def run_smc(self, no_samples: int):
        samples = self._generate_initial_states(no_samples)
        assert samples.shape == (no_samples, self.__n * self.__n)
        
        self.smc.reset(samples = samples)
        lam_list, diff_log_z = self.smc.build_intermediate_dists(max_steps=64, n_bisect=30, mcmc_iters=50)
        self.log_z += diff_log_z
        return lam_list

    def get_current_sample_list(self):
        return self.smc.get_current_sample_list()

    def est_log_total(self):
        return self.log_z

#### Latin Square Solver

Secondly, define the prior/target dist logpdfs we will sample from as well as the proposed_fn to move to the next state


In [ ]:
import jax
import jax.numpy as jnp
from jax import random

import numpy as np 

def prior_dist_logpdf(flat_board: jnp.array):
    """
    Supports:
        flat_board.shape == (N,)
        flat_board.shape == (B, N)
    """
    board_size = flat_board.shape[-1]
    n = int(np.sqrt(board_size))

    log_prob = -n * jnp.log(jnp.arange(1, n + 1)).sum()

    if flat_board.ndim == 1:
        return log_prob
    else:
        return jnp.full((flat_board.shape[0],), log_prob)

def target_dist_logpdf(flat_board: jnp.array, score_fn, lam=1000):
    """
    Supports:
        (N,)  -> scalar
        (B,N) -> (B,)
    """
    if flat_board.ndim == 1:
        return -lam * score_fn(flat_board)
    else:
        scores = jax.vmap(score_fn)(flat_board)
        return -lam * scores

In [43]:
import jax
import jax.numpy as jnp
from jax import random
import numpy as np

def proposed_fn(flat_board: jnp.array, key):
    """
    Supports:
        (N,)
        (B,N)
    """

    def propose_one(board: np.array, key):
        board_size = board.shape[0]
        n = jnp.sqrt(board_size).astype(int)

        key1, key2 = random.split(key)

        row = random.randint(key1, (), 0, n)
        col = random.randint(key2, (), 0, n - 1)

        i = row * n + col
        j = i + 1

        board_i, board_j = board[i], board[j]
        board = board.at[i].set(board_j)
        board = board.at[j].set(board_i)
        return board 

    if flat_board.ndim == 1:
        return propose_one(flat_board, key)

    else:
        keys = random.split(key, flat_board.shape[0])
        return jax.vmap(propose_one)(flat_board, keys)

In [44]:
from jax import random, vmap
from jax import numpy as jnp 
from functools import partial

class LatinSquareSampler(ISolver): 
    def __init__(self, n: int, score_fn):
        self.__n = n 
        super().__init__(
            n = n, 
            target_dist_logpdf = partial(target_dist_logpdf, score_fn = score_fn), 
            prior_dist_logpdf = prior_dist_logpdf, 
            proposed_fn = proposed_fn
        )

    def _generate_initial_state(self, key):
        sub_keys = random.split(key, num = self.__n)
        perm = vmap(
            lambda key: random.permutation(key, jnp.arange(1, self.__n + 1)) 
        )(sub_keys)
        return perm.reshape(-1)  

Now let's test our solver!

In [45]:
from jax import numpy as jnp

grouth_truth = [
    1, 
    2, 
    12, 
    576, 
    161280, 
    812851200, 
    61479419904000, 
    108776032459082956800, 
    5524751496156892842531225600, 
    9982437658213039871725064756920320000, 
    776966836171770144107444346734230682311065600000
]

log_grouth_truth = [
    0.0,
    0.6931471805599453,
    2.4849066497880004,
    6.356107660695891,
    11.989476363991853,
    20.51524607852745,
    31.74754634896515,
    46.13891007716285,
    63.87674183985093,
    85.1927379585188,
    59.61662636289668,
]

In [ ]:
NDIMS = 9
latin_solver = LatinSquareSampler(
    n = NDIMS, 
    score_fn = Scorer.score_repetitive_columns
)

print(f"before log_z: {latin_solver.log_z}")

lam_list = latin_solver.run_smc(
    no_samples = 300_000
)

print(f"aft log_z: {latin_solver.log_z}")

samples = latin_solver.get_current_sample_list()

before log_z: 0.0


Display some Latin square samples 

In [ ]:
for sample in samples[:10]:
    print(sample.reshape(NDIMS, NDIMS), "\n")

In [ ]:
latin_solver.est_log_total()

In [ ]:
from jax import vmap, jit
from jax import numpy as jnp
import numpy as np

def solve(dims: int):
    latin_solver = LatinSquareSampler(n = dims, score_fn = Scorer.score_repetitive_columns)
    latin_solver.run_smc(no_samples = 100_000)
    return latin_solver.est_log_total() 

ans = [solve(dims) for dims in range(1, 12)]

In [ ]:
for dims in range(1, 12):
    print(f"est log value = {ans[dims - 1]}, log_grouth_truth = {log_grouth_truth[dims - 1]}")

#### Sudoku Solver

In [4]:
import jax
import jax.numpy as jnp
from jax import random

import numpy as np

def target_dist_logpdf(flat_board: jnp.array, score_fn, lam=4000):
    """
    Supports:
        (N,)  -> scalar
        (B,N) -> (B,)
    """
    if flat_board.ndim == 1:
        return -lam * score_fn(flat_board)
    else:
        scores = jax.vmap(score_fn)(flat_board)
        return -lam * scores


In [5]:
from jax import random
from jax import numpy as jnp
from functools import partial
import jax
import numpy as np


class SudokuSolver(ISolver):
    """Sudoku SMC solver with block-wise prior / proposal.

    Init:
        For each of the 9 3x3 blocks, permute the missing values {1..9}\\filled
        into the originally empty cells. Every block always contains 1..9.

    Proposal:
        Pick one 3x3 block at random, then swap two originally empty cells
        inside that block (no-op if the block has < 2 empty cells).
    """

    def __init__(self, configuration: jnp.array, score_fn):
        self.__n = 9
        self.configuration = jnp.asarray(configuration).reshape(9, 9)

        all_values = set(range(1, 10))
        # Per-block originally empty flat indices and the values they must hold
        self.block_unfilled_idx = []   # list[jnp.ndarray], length 9
        self.block_missing_vals = []   # list[jnp.ndarray], length 9

        log_fact = [0.0]
        for k in range(1, 10):
            log_fact.append(log_fact[-1] + float(np.log(k)))

        log_prior = 0.0
        for b in range(9):
            br, bc = divmod(b, 3)
            filled = set()
            empty_idx = []
            for dr in range(3):
                for dc in range(3):
                    r, c = 3 * br + dr, 3 * bc + dc
                    v = int(self.configuration[r, c])
                    if v == 0:
                        empty_idx.append(r * 9 + c)
                    else:
                        filled.add(v)
            missing = sorted(all_values - filled)
            assert len(missing) == len(empty_idx), (
                f"block {b}: {len(missing)} missing vs {len(empty_idx)} empty"
            )
            self.block_unfilled_idx.append(jnp.array(empty_idx, dtype=jnp.int32))
            self.block_missing_vals.append(jnp.array(missing, dtype=jnp.int32))
            log_prior -= log_fact[len(empty_idx)]

        self._log_prior_const = float(log_prior)

        # Pad to fixed shape so JAX vmap can index by block id
        max_k = max(len(x) for x in self.block_unfilled_idx)
        self._pad_idx = jnp.full((9, max_k), -1, dtype=jnp.int32)
        self._pad_k = jnp.zeros((9,), dtype=jnp.int32)
        for b, idx in enumerate(self.block_unfilled_idx):
            k = int(idx.shape[0])
            self._pad_k = self._pad_k.at[b].set(k)
            if k > 0:
                self._pad_idx = self._pad_idx.at[b, :k].set(idx)

        def prior_dist_logpdf(flat_board):
            # Uniform over block completions: constant on the support
            if flat_board.ndim == 1:
                return jnp.asarray(self._log_prior_const)
            return jnp.full((flat_board.shape[0],), self._log_prior_const)

        super().__init__(
            n=9,
            target_dist_logpdf=partial(target_dist_logpdf, score_fn=score_fn),
            prior_dist_logpdf=prior_dist_logpdf,
            proposed_fn=self.proposed_fn,
        )

    def _generate_initial_state(self, key):
        board = self.configuration.reshape(-1)
        keys = random.split(key, 9)
        for b in range(9):
            idx = self.block_unfilled_idx[b]
            vals = self.block_missing_vals[b]
            k = int(idx.shape[0])
            if k == 0:
                continue
            permuted = random.permutation(keys[b], vals)
            board = board.at[idx].set(permuted)
        return board

    def proposed_fn(self, flat_board, key):
        pad_idx = self._pad_idx
        pad_k = self._pad_k

        def propose_one(board, key):
            key_b, key_ij = random.split(key)
            # choose a block that has at least 2 originally empty cells when possible
            eligible = pad_k >= 2
            # fallback: if somehow none eligible, pick any block (swap becomes no-op)
            weights = eligible.astype(jnp.float32) + 1e-8
            b = random.choice(key_b, 9, p=weights / weights.sum())
            k = pad_k[b]
            idx = pad_idx[b]

            # sample two distinct positions among the first k entries of idx
            # (safe when k>=2; when k<2 we just return board unchanged)
            def do_swap(board):
                # choose 2 distinct integers in [0, k)
                # use random.choice without replacement on arange(k) via permutation
                perm = random.permutation(key_ij, jnp.arange(idx.shape[0]))
                # only the first k slots are valid; take two from those by masking
                # Build a length-k permutation via argsort of random keys on valid slots
                noise = random.uniform(key_ij, (idx.shape[0],))
                # invalidate padded slots so they sort last
                noise = jnp.where(jnp.arange(idx.shape[0]) < k, noise, 2.0)
                order = jnp.argsort(noise)
                i0, i1 = order[0], order[1]
                p0, p1 = idx[i0], idx[i1]
                v0, v1 = board[p0], board[p1]
                board = board.at[p0].set(v1)
                board = board.at[p1].set(v0)
                return board

            return jax.lax.cond(k >= 2, do_swap, lambda board: board, board)

        if flat_board.ndim == 1:
            return propose_one(flat_board, key)

        keys = random.split(key, flat_board.shape[0])
        return jax.vmap(propose_one)(flat_board, keys)


Now let's try our solver with some puzzle!

In [11]:
from jax import numpy as jnp

# https://www.nytimes.com/puzzles/sudoku/hard
# 25/08/2026
problem = jnp.array([
    0,0,9,0,6,0,0,0,0,
    5,0,0,4,0,0,7,3,0,
    0,0,3,0,0,2,4,0,0,
    0,0,0,8,1,0,0,0,0,
    0,1,0,0,9,0,5,0,6,
    0,0,0,0,0,6,0,0,7,
    9,0,5,0,0,0,0,0,0,
    0,2,4,6,0,0,0,1,8, 
    0,0,0,0,0,0,0,7,0,
])

sudoku_solver = SudokuSolver( 
    configuration = problem,
    score_fn = Scorer.score_sudoku
)

In [12]:
lam_list = sudoku_solver.run_smc(
    no_samples = 10_000
)

samples = sudoku_solver.get_current_sample_list()

In [13]:
sudoku_solver.est_log_total()

Array(-4.8244324, dtype=float32)

In [14]:
from tqdm import tqdm
for sample in samples[:10]:
    print(Scorer.score_sudoku(sample))

0
0
0
0
0
0
0
0
0
0


In [15]:
samples[0].reshape(9,9)

Array([[1, 4, 9, 7, 6, 3, 8, 5, 2],
       [5, 6, 2, 4, 8, 1, 7, 3, 9],
       [8, 7, 3, 9, 5, 2, 4, 6, 1],
       [4, 5, 6, 8, 1, 7, 2, 9, 3],
       [2, 1, 7, 3, 9, 4, 5, 8, 6],
       [3, 9, 8, 5, 2, 6, 1, 4, 7],
       [9, 3, 5, 1, 7, 8, 6, 2, 4],
       [7, 2, 4, 6, 3, 5, 9, 1, 8],
       [6, 8, 1, 2, 4, 9, 3, 7, 5]], dtype=int32)